# Analyzing McIDAS-V output back in numpy

In [ ]:
%load_ext mcidasv_jupyter
%mcv_connect /path/to/runMcV
%mcv_replay off

## Render a real GOES image and grab the captured PNG path

In [ ]:
import mcidasv_jupyter as mcv
import numpy as np
session = mcv.get_session()

result = session.run('''
data = loadADDEImage(server='adde.ucar.edu', dataset='EAST',
                     descriptor='CONUSC13', size='ALL', unit='TEMP')
panel = buildWindow(height=600, width=800)
layer = panel[0].createLayer('Image Display', data)
panel[0].setWireframe(False)
layer.setEnhancement('ABI IR Temperature', range=(200, 320))
''')
print('captured image:', result.image)

## Read the image into numpy and analyze it

In [ ]:
import matplotlib.pyplot as plt

img = plt.imread(result.image)
print('image array', img.shape, img.dtype)

gray = img[..., :3].mean(axis=2)
print('mean luminance', float(gray.mean()))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(img); ax[0].set_title('McIDAS-V capture'); ax[0].axis('off')
ax[1].hist(gray.ravel(), bins=60, color='steelblue')
ax[1].set_title('luminance histogram'); plt.tight_layout()

## Close the loop: threshold in numpy, send a mask back to McIDAS-V

In [ ]:
from math import ceil
step = max(1, ceil(max(gray.shape) / 160))
small = gray[::step, ::step]
mask = (small > np.percentile(small, 90)).astype('f4')

glats = np.linspace(50, 20, small.shape[0])
glons = np.linspace(-125, -66, small.shape[1])

session.run('''
panel = buildWindow(height=600, width=800)
layer = panel[0].createLayer('Color-Shaded Plan View', g)
panel[0].setProjection('US>CONUS')
layer.setLayerLabel(label='numpy mask: brightest 10% pixels')
''', arrays={'g': (mask, glats, glons)})